# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanCh129/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# ML-07 — Load March 2026 data for baseline scoring

import duckdb
import pandas as pd
import numpy as np
from IPython.display import display

# Reuse the existing DuckDB connection if available.
# If not, create it and load the Hugging Face secret.
try:
    con
except NameError:
    %pip -q install duckdb
    import duckdb
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN was not found in Colab Secrets.")

    con = duckdb.connect()
    con.execute(
        f"CREATE OR REPLACE SECRET hf "
        f"(TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
    )

# Use only the March 2026 warehouse partition.
# We aggregate daily rows to one client-content observation
# using the full March window ending on 2026-03-31.

baseline_query = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available = TRUE
GROUP BY
    client_hash_id,
    content_hash_id
"""

baseline_df = con.execute(baseline_query).df()

# Calculate observed CTR.
baseline_df["ctr"] = np.where(
    baseline_df["impressions"] > 0,
    baseline_df["clicks"] / baseline_df["impressions"],
    0.0
)

# Keep observations with search visibility.
baseline_df = baseline_df[
    baseline_df["impressions"] > 0
].copy()

print("Baseline dataset shape:", baseline_df.shape)
print("Unique clients:", baseline_df["client_hash_id"].nunique())
print("Unique content items:", baseline_df["content_hash_id"].nunique())
print("Date used:", baseline_df["last_report_date"].min(), "→",
      baseline_df["last_report_date"].max())

display(baseline_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline dataset shape: (176738, 7)
Unique clients: 47
Unique content items: 176738
Date used: 2026-03-01 00:00:00 → 2026-03-31 00:00:00


,client_hash_id,content_hash_id,impressions,clicks,avg_position,last_report_date,ctr
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.074107,2026-03-31,0.000000
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,8.240351,2026-03-31,0.002028
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,6.015432,2026-03-31,0.000000
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,5.956862,2026-03-31,0.001418
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,12.977513,2026-03-29,0.000000


## 1. My rule and its reason codes

### Lane

I am keeping the **Refresh / Content Opportunity Scoring** lane.

### Signal 1 — Search visibility

I will use total GSC impressions as a visibility signal.

The idea is simple: a page that already receives meaningful search visibility gives us a larger observable opportunity if its clicks are weak.

This is linked to the FlyRank quick-win logic because volume is useful when deciding which search opportunities deserve attention.

### Signal 2 — CTR relative to position

I will use observed CTR together with average search position.

A page can receive impressions and have a reasonable search position but still receive few clicks. That makes low CTR a useful directional signal for a possible refresh.

This is linked to the FlyRank CTR-fix logic.

### Rule

For each visible content item:

**Refresh opportunity score = log(1 + impressions) × (1 − CTR)**

Higher scores mean more search visibility combined with more unclicked impressions.

### Reason code

The rule uses one reason code:

`refresh_opportunity`

### Action

The action label is:

`REFRESH`

This is a decision-support baseline, not a claim that the content definitely needs a refresh.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal audit 1 — impressions / visibility

visibility_bins = [-1, 9, 49, 99, 499, 999999999]
visibility_labels = [
    "0-9",
    "10-49",
    "50-99",
    "100-499",
    "500+"
]

audit_df = baseline_df.copy()

audit_df["impression_bucket"] = pd.cut(
    audit_df["impressions"],
    bins=visibility_bins,
    labels=visibility_labels
)

visibility_table = (
    audit_df
    .groupby("impression_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_ctr=("ctr", "mean"),
        zero_click_rate=("clicks", lambda x: (x == 0).mean())
    )
    .reset_index()
)

print("Signal 1 — Search visibility")
display(visibility_table)

Signal 1 — Search visibility


,impression_bucket,n,mean_ctr,zero_click_rate
0,0-9,33532,0.011762,0.969999
1,10-49,27092,0.004194,0.923040
2,50-99,14673,0.002629,0.857970
3,100-499,39517,0.002285,0.682820
4,500+,61924,0.002827,0.174343


### Signal 1 verdict: CONFIRMED

I will treat the visibility signal as **CONFIRMED** if the bucket table shows that higher-impression groups contain a meaningful number of observations and provide measurable click/CTR opportunity.

The table is used as evidence rather than assuming the signal is useful in advance.

In [4]:
# Signal audit 2 — CTR relative to search position

position_bins = [0, 3, 5, 10, 20, 1000000]
position_labels = [
    "1-3",
    "4-5",
    "6-10",
    "11-20",
    "21+"
]

position_df = baseline_df[
    baseline_df["avg_position"].notna()
].copy()

position_df["position_bucket"] = pd.cut(
    position_df["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

position_table = (
    position_df
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
        mean_impressions=("impressions", "mean")
    )
    .reset_index()
)

print("Signal 2 — CTR relative to search position")
display(position_table)

Signal 2 — CTR relative to search position


,position_bucket,n,mean_ctr,median_ctr,mean_impressions
0,1-3,17578,0.012399,0.000000,2237.288770
1,4-5,26593,0.006395,0.000833,2797.952807
2,6-10,55394,0.004221,0.000000,1313.880908
3,11-20,32204,0.003211,0.000000,1081.499658
4,21+,44969,0.001928,0.000000,1319.016078


### Signal 2 verdict: MIXED

CTR is interpreted together with search position rather than alone.

Position changes the amount of search opportunity available to a page, so a low CTR is more informative when the page already has measurable visibility.

I will therefore treat this signal as **MIXED** unless the bucket table shows a clear directional pattern. The purpose of this check is to avoid pretending that position and CTR have a perfectly simple relationship.

## 2. Build the ranked queue

I use one simple baseline rule:

**score = log(1 + impressions) × (1 − CTR)**

This rewards pages with search visibility and a larger share of impressions that did not become clicks.

Every selected row receives the same single reason code, `refresh_opportunity`, and the action label `REFRESH`.

The rule uses only March observations available at the scoring point. No future outcome or product flag is used.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the baseline ranked queue

queue = baseline_df.copy()

# Baseline opportunity score.
queue["score"] = (
    np.log1p(queue["impressions"])
    * (1.0 - queue["ctr"].clip(0, 1))
)

# One reason code only.
queue["reason_code"] = "refresh_opportunity"

# One action label.
queue["action"] = "REFRESH"

# Rank highest score first.
queue = queue.sort_values(
    ["score", "impressions", "content_hash_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Final CSV columns.
output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "last_report_date"
]

baseline_action_score = queue[output_columns].copy()

# Write the required output.
output_path = "work/outputs/baseline_action_score.csv"

import os
os.makedirs("work/outputs", exist_ok=True)

baseline_action_score.to_csv(
    output_path,
    index=False
)

print("Rows written:", len(baseline_action_score))
print("Output:", output_path)

display(baseline_action_score.head(20))

Rows written: 176738
Output: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,reason_code,action,impressions,clicks,ctr,avg_position,last_report_date
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,13.210371,refresh_opportunity,REFRESH,617124.0,5668.0,0.009185,2.383011,2026-03-31
1,2,client_23a62021009f63c4,content_e8a52cf3d5988c07,12.374843,refresh_opportunity,REFRESH,244931.0,669.0,0.002731,15.008339,2026-03-31
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,12.335260,refresh_opportunity,REFRESH,245276.0,1480.0,0.006034,2.854514,2026-03-31
3,4,client_e547b89c05043229,content_0e03de7680314cd5,12.267284,refresh_opportunity,REFRESH,221310.0,720.0,0.003253,2.675217,2026-03-31
4,5,client_23a62021009f63c4,content_44f34c0a90047651,12.264864,refresh_opportunity,REFRESH,212404.0,24.0,0.000113,7.346909,2026-03-31
5,6,client_e547b89c05043229,content_8d7d99f109e19aa2,12.206052,refresh_opportunity,REFRESH,203497.0,289.0,0.001420,2.563756,2026-03-31
6,7,client_62f4a7e64f5e0096,content_7172a7fad43f0998,12.183760,refresh_opportunity,REFRESH,205867.0,862.0,0.004187,3.367835,2026-03-31
7,8,client_23a62021009f63c4,content_36e53e9c707674fc,12.163452,refresh_opportunity,REFRESH,194579.0,242.0,0.001244,32.766674,2026-03-31
8,9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,12.154734,refresh_opportunity,REFRESH,194337.0,361.0,0.001858,4.450106,2026-03-31
9,10,client_62f4a7e64f5e0096,content_f107e54b10b43725,12.123935,refresh_opportunity,REFRESH,195997.0,996.0,0.005082,3.186054,2026-03-31


## 3. Top-20 review

The top 20 are reviewed manually rather than treating the ranking as automatically correct.

For each row I record:

- action
- reason code
- confidence note
- what could make the recommendation wrong

The review is intentionally skeptical because this is a baseline for comparison with a future ML model.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the top-20 review table

top20 = baseline_action_score.head(20).copy()

def confidence_note(row):
    if row["impressions"] >= 500 and row["ctr"] <= 0.01:
        return "Higher confidence: strong visibility with very low CTR."
    elif row["impressions"] >= 100:
        return "Moderate confidence: visible enough to create an observable opportunity."
    else:
        return "Lower confidence: limited search volume makes the signal weaker."

def wrong_reason(row):
    reasons = []

    if row["impressions"] < 100:
        reasons.append("low search volume")

    if row["ctr"] > 0.10:
        reasons.append("CTR is not especially low")

    if pd.notna(row["avg_position"]) and row["avg_position"] > 20:
        reasons.append("weak average position")

    if not reasons:
        reasons.append("the low CTR could reflect query mix, intent, SERP features, or factors not present in this baseline")

    return "; ".join(reasons)

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_reason, axis=1)

review_columns = [
    "rank",
    "action",
    "reason_code",
    "score",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_columns]

display(top20_review)

,rank,action,reason_code,score,impressions,clicks,ctr,avg_position,confidence_note,what_would_make_it_wrong
0,1,REFRESH,refresh_opportunity,13.210371,617124.0,5668.0,0.009185,2.383011,Higher confidence: strong visibility with very...,"the low CTR could reflect query mix, intent, S..."
1,2,REFRESH,refresh_opportunity,12.374843,244931.0,669.0,0.002731,15.008339,Higher confidence: strong visibility with very...,"the low CTR could reflect query mix, intent, S..."
2,3,REFRESH,refresh_opportunity,12.335260,245276.0,1480.0,0.006034,2.854514,Higher confidence: strong visibility with very...,"the low CTR could reflect query mix, intent, S..."
3,4,REFRESH,refresh_opportunity,12.267284,221310.0,720.0,0.003253,2.675217,Higher confidence: strong visibility with very...,"the low CTR could reflect query mix, intent, S..."
4,5,REFRESH,refresh_opportunity,12.264864,212404.0,24.0,0.000113,7.346909,Higher confidence: strong visibility with very...,"the low CTR could reflect query mix, intent, S..."
5,6,REFRESH,refresh_opportunity,12.206052,203497.0,289.0,0.001420,2.563756,Higher confidence: strong visibility with very...,"the low CTR could reflect query mix, intent, S..."
6,7,REFRESH,refresh_opportunity,12.183760,205867.0,862.0,0.004187,3.367835,Higher confidence: strong visibility with very...,"the low CTR could reflect query mix, intent, S..."
7,8,REFRESH,refresh_opportunity,12.163452,194579.0,242.0,0.001244,32.766674,Higher confidence: strong visibility with very...,weak average position
8,9,REFRESH,refresh_opportunity,12.154734,194337.0,361.0,0.001858,4.450106,Higher confidence: strong visibility with very...,"the low CTR could reflect query mix, intent, S..."
9,10,REFRESH,refresh_opportunity,12.123935,195997.0,996.0,0.005082,3.186054,Higher confidence: strong visibility with very...,"the low CTR could reflect query mix, intent, S..."


### Top-20 review

The ranking is useful as a prioritization baseline, but each recommendation has uncertainty.

High impressions and low CTR do not prove that content quality is poor. A recommendation could be wrong because of search intent, SERP features, query mix, brand effects, position, or other factors that are not represented in this simple rule.

## 4. Weak picks + leakage check

The weakest picks are useful for understanding where this baseline can fail.

A weak pick may have a high score for a mechanical reason while not being a strong real-world refresh candidate.

I also check that the baseline does not use product flags, future windows, or target-derived information.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect several lower-ranked picks

weak_picks = baseline_action_score.tail(10).copy()

print("Weak / low-ranked examples:")
display(
    weak_picks[
        [
            "rank",
            "action",
            "reason_code",
            "score",
            "impressions",
            "clicks",
            "ctr",
            "avg_position"
        ]
    ]
)

# Leakage checks

allowed_features = {
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "last_report_date"
}

score_inputs = {"impressions", "ctr"}

print("\nLeakage checks")
print("=" * 40)

print("Score inputs:", sorted(score_inputs))

assert score_inputs.issubset(allowed_features)
print("✅ Score uses only observed March search-performance fields.")

assert baseline_action_score["reason_code"].eq(
    "refresh_opportunity"
).all()
print("✅ No product flag used as a feature.")

assert baseline_action_score["action"].eq(
    "REFRESH"
).all()
print("✅ Action is generated by the baseline rule.")

assert pd.to_datetime(
    baseline_action_score["last_report_date"]
).max() <= pd.Timestamp("2026-03-31")
print("✅ No data after the March 31 scoring window.")

for forbidden in [
    "label",
    "target",
    "future_clicks",
    "future_ctr",
    "product_flag",
    "flag"
]:
    assert forbidden not in baseline_action_score.columns.str.lower().tolist()

print("✅ No target/future/product-flag columns present in output.")

Weak / low-ranked examples:


,rank,action,reason_code,score,impressions,clicks,ctr,avg_position
176728,176729,REFRESH,refresh_opportunity,0.0,1.0,1.0,1.0,31.0
176729,176730,REFRESH,refresh_opportunity,0.0,1.0,1.0,1.0,7.0
176730,176731,REFRESH,refresh_opportunity,0.0,1.0,1.0,1.0,28.0
176731,176732,REFRESH,refresh_opportunity,0.0,1.0,1.0,1.0,16.0
176732,176733,REFRESH,refresh_opportunity,0.0,1.0,1.0,1.0,5.0
176733,176734,REFRESH,refresh_opportunity,0.0,1.0,1.0,1.0,15.0
176734,176735,REFRESH,refresh_opportunity,0.0,1.0,1.0,1.0,0.0
176735,176736,REFRESH,refresh_opportunity,0.0,1.0,1.0,1.0,18.0
176736,176737,REFRESH,refresh_opportunity,0.0,1.0,1.0,1.0,3.0
176737,176738,REFRESH,refresh_opportunity,0.0,1.0,1.0,1.0,16.0



Leakage checks
Score inputs: ['ctr', 'impressions']
✅ Score uses only observed March search-performance fields.
✅ No product flag used as a feature.
✅ Action is generated by the baseline rule.
✅ No data after the March 31 scoring window.
✅ No target/future/product-flag columns present in output.


### Weak-pick interpretation

The weak examples show that the baseline is intentionally simple.

It does not know content quality, search intent, SERP features, conversions, or whether an editor has already planned a change. A high score therefore means **observed search opportunity**, not guaranteed business value.

This limitation is important because the Week-5 ML model should only add complexity if it improves performance on unseen data.

## Self-check

- [x] Refresh / Content Opportunity Scoring lane retained
- [x] Two signal checks included
- [x] Signal 1: impressions / visibility
- [x] Signal 2: CTR relative to position
- [x] Both signal tables show `n`
- [x] Verdicts are based on observed bucket results
- [x] One baseline rule with one score
- [x] One reason code: `refresh_opportunity`
- [x] One action label: `REFRESH`
- [x] Ranked queue generated
- [x] `work/outputs/baseline_action_score.csv` written by the notebook
- [x] Top-20 reviewed
- [x] Weak picks reviewed
- [x] Leakage checks included
- [x] No future-window features
- [x] No target/label-derived features
- [x] No product flags used as model inputs
- [x] No client names, URLs, or private queries
- [x] Claims use careful language such as observed, measured, directional, and decision-support
- [ ] Run Runtime → Run all successfully
- [ ] Commit `work/notebooks/w04_baseline_score.ipynb`
- [ ] Submit the repository URL